#  Split and Balance 



In [1]:
import json, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

ROOT = Path.cwd() if (Path.cwd() / "model").is_dir() else Path.cwd().parent  # repository root
sys.path.insert(0, str(ROOT / "model"))
from split_balance import AMBIGUOUS, DERM1M_GROUPS, SPLITS, assign_splits, group_keys, leakage, map_labels

# Derm1M_v2_pretrain.csv: https://huggingface.co/datasets/redlessone/Derm1M (gated: accept the terms first)
MANIFEST = ROOT / "Derm1M_v2_pretrain.csv"
if not MANIFEST.exists():
    from huggingface_hub import hf_hub_download
    MANIFEST = Path(hf_hub_download("redlessone/Derm1M", "Derm1M_v2_pretrain.csv", repo_type="dataset"))

config = json.loads((ROOT / "model" / "derm1m_classes.json").read_text(encoding="utf-8"))
SEED, RATIOS, MIN_COUNT = 42, (0.70, 0.15, 0.15), 500

## 1. Load the data and check duplicate records

In [2]:
raw = pd.read_csv(MANIFEST, usecols=["filename", "disease_label", "source"], encoding="utf-8-sig")
conflicts = raw.groupby("filename").disease_label.nunique().loc[lambda s: s > 1].index
print(f"rows: {len(raw):,} | duplicate records: {raw.duplicated().sum():,} | images with conflicting labels: {len(conflicts):,}")

raw = raw.drop_duplicates()
raw = raw[~raw.filename.isin(conflicts)].drop_duplicates("filename")

rows: 413,210 | duplicate records: 0 | images with conflicting labels: 0


## 2. Map labels to the selected classes

`disease_label` is a comma-separated list of terms. An image is kept only if every term maps to the same class
in `model/derm1m_classes.json` (`dermatitis` and `nail disease` are ignored). Other labels are excluded.

In [3]:
df = raw.assign(label=map_labels(raw.disease_label, config))
print(f"outside the selected classes: {df.label.isna().sum():,} | spanning several classes: {(df.label == AMBIGUOUS).sum():,}")

counts = df.label.value_counts()
kept = [c for c in config["classes"] if counts.get(c, 0) >= MIN_COUNT]
df = df[df.label.isin(kept)].reset_index(drop=True)
df.label.value_counts().reindex(kept).to_frame("images")

outside the selected classes: 359,030 | spanning several classes: 483


,images
label,
acne,2745
eczema,10387
psoriasis,14644
melanoma,8100
benign_nevus,10760
fungal_infection,7061


## 3. Group related images

Images from the same YouTube video, IIYI case, Twitter/Reddit post, PubMed article or textbook page share a group.
`public` sources have no identifier in the file name, so each of their images is its own group.

In [4]:
df["group_id"] = group_keys(df.filename, DERM1M_GROUPS)
ungrouped = df.group_id == df.filename
print(f"groups: {df.group_id.nunique():,} | ungrouped images: {ungrouped.mean():.1%}")

pd.DataFrame({"groups": df.groupby("label").group_id.nunique(),
              "ungrouped %": (ungrouped.groupby(df.label).mean() * 100).round(1)}).reindex(kept)

groups: 24,590 | ungrouped images: 24.3%


,groups,ungrouped %
label,,
acne,1197,4.3
eczema,6024,31.1
psoriasis,3865,1.6
melanoma,3521,28.1
benign_nevus,7112,49.1
fungal_infection,4173,27.1


## 4. Stratified, grouped split and leakage check

`StratifiedGroupKFold` with 20 folds of 5 %: folds 0–2 → test, 3–5 → val, the rest → train.
Every count below must be 0.

In [5]:
df["split"] = assign_splits(df, RATIOS, SEED)
leak = leakage(df)
assert not any(v for pair in leak.values() for v in pair.values()), leak
pd.DataFrame(leak).T

,group_id,filename
train vs val,0,0
train vs test,0,0
val vs test,0,0


## 5. Class distribution per split

In [6]:
dist = pd.crosstab(df.label, df.split).reindex(index=kept, columns=list(SPLITS))
pd.concat([dist, (dist / dist.sum() * 100).round(1).add_suffix(" %")], axis=1)

split,train,val,test,train %,val %,test %
label,,,,,,
acne,1922,411,412,5.1,5.1,5.1
eczema,7271,1557,1559,19.3,19.3,19.3
psoriasis,10248,2198,2198,27.3,27.3,27.3
melanoma,5670,1215,1215,15.1,15.1,15.1
benign_nevus,7532,1614,1614,20.0,20.0,20.0
fungal_infection,4943,1059,1059,13.2,13.1,13.1


## 6. Class imbalance

Weights are computed on the training split only and passed to `CrossEntropyLoss(weight=...)`.
The comparison with weighted sampling, oversampling and undersampling is in `docs/data_split.md`.

In [7]:
train_counts = df.label[df.split == "train"].value_counts()
print(f"imbalance ratio in train (largest / smallest class): {train_counts.max() / train_counts.min():.2f}")

weights = compute_class_weight("balanced", classes=np.array(kept), y=df.label[df.split == "train"])
pd.DataFrame({"train images": train_counts.reindex(kept), "weight": weights.round(4)}, index=kept)

imbalance ratio in train (largest / smallest class): 5.33


,train images,weight
acne,1922,3.2593
eczema,7271,0.8616
psoriasis,10248,0.6113
melanoma,5670,1.1048
benign_nevus,7532,0.8317
fungal_infection,4943,1.2673


## 7. Save the outputs

Runs the script, which writes the manifests and reports to `dataset/splits/`, then confirms the saved split
matches this notebook.

In [8]:
run = subprocess.run([sys.executable, "model/split_balance.py", "--manifest", str(MANIFEST)],
                     cwd=ROOT, capture_output=True, text=True, check=True)
print(run.stdout)

saved = pd.concat(pd.read_csv(ROOT / "dataset" / "splits" / f"{s}.csv").assign(split=s) for s in SPLITS)
merged = df.merge(saved, on="filename", suffixes=("", "_saved"))
assert len(merged) == len(df) == len(saved) and (merged.split == merged.split_saved).all()
print("Saved files match the split above.")

Selected 53,697 of 413,210 rows in 6 classes
train 37,586 | val 8,054 | test 8,057 | leakage checks passed
Report: dataset/splits/SPLIT_REPORT.md

Saved files match the split above.
